In [1]:
import torch
import string

In [2]:
words = open('names.txt', 'r').read().splitlines()

In [3]:
words[:10]

['emma',
 'olivia',
 'ava',
 'isabella',
 'sophia',
 'charlotte',
 'mia',
 'amelia',
 'harper',
 'evelyn']

In [48]:
cleaned_words = []

for word in words:
    cleaned_words.append('..' + word + '.')

cleaned_words[:10]

['..emma.',
 '..olivia.',
 '..ava.',
 '..isabella.',
 '..sophia.',
 '..charlotte.',
 '..mia.',
 '..amelia.',
 '..harper.',
 '..evelyn.']

In [49]:
for w in cleaned_words[:1]:
    for i in range(2, len(w)):
        print(w[i-2], w[i-1], w[i])

. . e
. e m
e m m
m m a
m a .


In [50]:
stoi = {c:i for i, c in enumerate(string.ascii_lowercase)} | {'.': 26}
itos = {i:c for i, c in enumerate(string.ascii_lowercase)} | {26: '.'}

In [51]:
xs = []
ys = []


for w in cleaned_words:
    for i in range(2, len(w)):
        xs.append(stoi[w[i-2]] * 27 + stoi[w[i-1]])
        ys.append(stoi[w[i]])

ys = torch.tensor(ys)
print(xs[:10])
print(ys[:10])

[728, 706, 120, 336, 324, 728, 716, 389, 305, 237]
tensor([ 4, 12, 12,  0, 26, 14, 11,  8, 21,  8])


In [52]:
import torch.nn.functional as F

xenc = F.one_hot(torch.tensor(xs), num_classes=27*27).float()
xenc[:10]

tensor([[0., 0., 0.,  ..., 0., 0., 1.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]])

In [53]:
xenc.shape

torch.Size([228146, 729])

In [72]:
g = torch.Generator().manual_seed(21)
W = torch.randn((729, 27), generator=g, requires_grad=True)

In [77]:
learning_rate = 50

# training loop
for k in range(200):

    # forward pass
    logits = xenc @ W
    counts = torch.exp(logits)
    prob = counts / counts.sum(1, keepdim=True)
    loss = - prob[torch.arange(len(xs)), ys].log().mean()

    # print loss
    print(loss.item())

    # backprop
    W.grad = None
    loss.backward()

    # update W
    W.data -= learning_rate * W.grad



2.253788709640503
2.253708600997925
2.253628730773926
2.253549098968506
2.253469467163086
2.253390073776245
2.2533111572265625
2.253232002258301
2.2531533241271973
2.253074884414673
2.2529966831207275
2.2529187202453613
2.252840518951416
2.252762794494629
2.252685308456421
2.252608060836792
2.252530813217163
2.2524538040161133
2.2523772716522217
2.252300500869751
2.2522242069244385
2.252148389816284
2.2520720958709717
2.2519962787628174
2.2519209384918213
2.251845359802246
2.251770257949829
2.251695156097412
2.251620054244995
2.2515456676483154
2.2514708042144775
2.251396656036377
2.2513225078582764
2.251248598098755
2.2511749267578125
2.25110125541687
2.251027822494507
2.2509543895721436
2.2508816719055176
2.2508084774017334
2.2507357597351074
2.2506635189056396
2.2505910396575928
2.250519037246704
2.2504467964172363
2.2503750324249268
2.250303268432617
2.250231981277466
2.2501604557037354
2.250089645385742
2.25001859664917
2.2499477863311768
2.2498769760131836
2.2498064041137695
2.24

In [78]:
g = torch.Generator().manual_seed(2147483647)

for _ in range(10):
    out = []
    ix1, ix2 = 26, 26                      # start context: '..'
    while True:
        idx = ix1 * 27 + ix2
        x = F.one_hot(torch.tensor([idx]), num_classes=27*27).float()
        logits = x @ W
        p = logits.exp()
        p = p / p.sum(1, keepdim=True)

        ix = torch.multinomial(p, num_samples=1, generator=g).item()
        if ix == 26:                       # end token
            break
        out.append(itos[ix])
        ix1, ix2 = ix2, ix
    print(''.join(out))


dey
na
khilvill
rilayian
marnk
oleyssleel
lia
sa
maydevikuston
puntand
